In [1]:
!rocm-smi




============================================ ROCm System Management Interface ============================================
====================================================== Concise Info ======================================================
Device  Node  IDs              Temp        Power     Partitions          SCLK    MCLK    Fan  Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Junction)  (Socket)  (Mem, Compute, ID)                                                  
0       1     0x74a1,   18420  42.0°C      138.0W    NPS1, SPX, 0        132Mhz  900Mhz  0%   auto  750.0W  0%     0%    
================================================== End of ROCm SMI Log ===================================================


In [2]:
!pip list

Package                 Version
----------------------- ---------------
absl-py                 2.3.1
accelerate              1.10.1
aiohappyeyeballs        2.6.1
aiohttp                 3.13.2
aiosignal               1.4.0
amdsmi                  26.1.0+5df6c765
anyio                   4.11.0
asttokens               3.0.1
async-timeout           5.0.1
attrs                   25.4.0
certifi                 2025.11.12
charset-normalizer      3.4.4
click                   8.1.8
cmake                   4.2.0
comm                    0.2.3
datasets                4.4.1
debugpy                 1.8.17
decorator               5.2.1
dill                    0.4.0
einops                  0.8.1
exceptiongroup          1.3.1
executing               2.2.1
filelock                3.19.1
flash-attn              2.8.3
frozenlist              1.8.0
fsspec                  2025.9.0
grpcio                  1.76.0
h11                     0.16.0
hf-xet                  1.2.0
httpcore                1.0.9
ht

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
import os
os.environ['HF_HOME'] = '/work1/lgarcia/pedrobpio/HF_files'
sys.path.append("..")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
from src.models.qwen3 import Qwen3

model_name = 'Qwen3-4B'
t = Qwen3(model_name = f'Qwen/{model_name}', device = "cuda:0")

t.model

INFO:root:Using device: cuda:0
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  3.47it/s]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [6]:
from src.datasets.lener import LenerDataset

l = LenerDataset(tokenizer=t.tokenizer)

In [7]:
data = l.load_dataset()
data

INFO:src.datasets.lener:Loading dataset: peluz/lener_br
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'peluz/lener_br' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'peluz/lener_br' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Using the latest cached version of the dataset since peluz/lener_br couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'lener_br' at /work1/lgarcia/pedrobpio/HF_files/datasets/peluz___lener_br/lener_br/1.0.0/4a8c97e6813b5c2d85a50faf0a3e6c24ea82f4a9044e6e9e8b24997d27399382 (last modified on Tue Nov 25 12:18:47 2025).
INFO:src.datasets.lener:Dataset loaded with splits: ['train', 'validation', 'test']
INFO:src.datasets.lener:Processing splits: ['train', 'validation', 'test']
INFO:src.datasets.lener:Original columns to remove after mapping: ['id', 'tokens', 'ner_tags']
INFO:src.datasets.lener:NER tag mapping created: {0: 'O', 1: 'B-ORGANIZACAO', 2: 'I-ORGANIZACAO', 3: 'B-PESSOA', 4: 'I-PESSOA', 5: 'B-TEMPO', 6: 'I-TEMPO', 7: 'B-LOCAL', 8: 'I-LOCAL', 9: 'B-LEGISLACAO', 10: 'I-LEGISLACAO', 11: 'B-JURISPRUDENCIA', 12: 'I-JURISPRUDENCIA'}
INFO:src.datasets.lener:Starting dataset mapping...
Map: 15656 examples [01:10, 110.91 examples/s]         
Map: 2354 ex

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1390
    })
})

In [21]:
from src.peft_configs.lora import LoraAdapter
# PRESET = 'baseline'
PRESET = 'full_attention'
# PRESET = 'all'
# PRESET = 'ffn'
# PRESET = 'full_attention_plus_ffn'
# PRESET = 'low_rank'
# PRESET = 'high_rank'
# PRESET = 'high_rank_XX'

lora = LoraAdapter(
    model=t.model,
    lora_preset = PRESET
)

model = lora.apply_lora()
model

INFO:root:Applying LoRA with preset: full_attention
INFO:root:LoRA configurations: {'r': 8, 'lora_alpha': 16, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'lora_dropout': 0.1, 'bias': 'none', 'task_type': <TaskType.CAUSAL_LM: 'CAUSAL_LM'>}


trainable params: 5,898,240 || all params: 4,028,366,336 || trainable%: 0.1464


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [22]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import torch
from trl import DataCollatorForCompletionOnlyLM
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "7"

# Training args
checkpoint_name = f'{model_name}_{PRESET}'
training_args = TrainingArguments(
    output_dir=f'./outputs/checkpoints/{checkpoint_name}',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    # gradient_accumulation_steps=8,
    # gradient_checkpointing=True, # Consider enabling this if memory is still an issue
    lr_scheduler_type='cosine',
    logging_steps=25,
    save_strategy="epoch",
    eval_strategy="no",
    learning_rate=2e-5,
    max_grad_norm=0.01,
    weight_decay=0.01,
    warmup_ratio=0.03,
    fp16=True,
    report_to="tensorboard",       # Enable TensorBoard logging
    logging_dir=f'./runs/my_logs/{checkpoint_name}',
)

response_template = "Resposta:\n"

response_template_ids = t.tokenizer.encode(
    response_template, 
    add_special_tokens=False
)

data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=t.tokenizer
)

# model.enable_input_require_grads() 

# 3. (Optional) Explicitly enable gradient checkpointing on the model instance
# model.gradient_checkpointing_enable()

# model.to('cuda:7')
# torch.cuda.set_device(7)
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=data["train"],
    tokenizer=t.tokenizer,
    data_collator=data_collator,
    
)

# Train
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/tmp/ipykernel_586849/445882028.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
25,0.851600
50,0.825400
75,0.601600
100,0.438700
125,0.247100
150,0.129100
175,0.061400
200,0.044400
225,0.052400
250,0.043900


TrainOutput(global_step=1957, training_loss=0.053981759556410686, metrics={'train_runtime': 1222.0353, 'train_samples_per_second': 6.406, 'train_steps_per_second': 1.601, 'total_flos': 3.5007655351576166e+17, 'train_loss': 0.053981759556410686, 'epoch': 1.0})

In [23]:
# 4. Crie seu prompt (TERMINANDO com o template de resposta)
prompt = (
    """Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.
- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.
- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.
- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      

segue o texto:
2 Documento assinado digitalmente conforme MP n° 2.200-2/2001 de 24/08/2001 , que institui a Infraestrutura de Chaves Públicas Brasileira - ICP-Brasil .
Resposta:\n 
"""
)
# A Curadoria Especial ofertou embargos à ação monitória , por negativa geral ( fl . 85 ) .
# Evidenciado que a citação somente foi aperfeiçoada após o decurso do prazo prescricional , tem-se por correta a extinção da demanda monitória , com resolução do mérito , nos termos do artigo 485 , inciso II , do Código de Processo Civil .
# 1 . Nos termos da Súmula nº 503 do colendo Superior Tribunal de Justiça , `` O prazo para ajuizamento de ação monitória em face do emitente de cheque sem força executiva é quinquenal , a contar do dia seguinte à data de emissão estampada na cártula '' .
# 5. Tokenize o prompt
inputs = t.tokenizer(prompt, return_tensors="pt").to('cuda:0')

# 6. Gere o texto
# pad_token_id é importante para evitar avisos
generation_config = {
    "do_sample": False,          # Habilita a amostragem
    "temperature": 0.01,         # Controla a "criatividade". Mais baixo = mais focado.
    "top_p": 0.9,               # Nucleus sampling: considera tokens até somarem 90% de prob.
    "repetition_penalty": 1, # Penaliza tokens que já apareceram (valores > 1.0)
    "max_new_tokens": 512,
    "pad_token_id": t.tokenizer.eos_token_id,
    "eos_token_id": t.tokenizer.eos_token_id # Garante que ele saiba quando parar
}

outputs = model.generate(
    **inputs,
    **generation_config
)

# 7. Decodifique a saída
generated_text = t.tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.
- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.
- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.
- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      

segue o texto:
2 Documento assinado digitalmente conforme MP n° 2.200-2/2001 de 24/08/2001 , que institui a Infraestrutura de Chaves Públicas Brasileira - ICP-Brasil .
Resposta:
 
2:O
Documento:O
assinado:O
digitalmente:O
conforme:O
MP:B-LEGISLACAO
n°:I-LEGISLACAO
2.200-2/

In [11]:
from src.scripts.utils import predict_entities_batch

texts = [
    "O ministério público acatou a decisão do STF e pediu a suspensão do processo contra o ex-presidente Lula, com base na Lei 17.681/2017.",
    "- Tratando-se de ação indenizatória ajuizada por pessoa incapaz , é obrigatória a intervenção do Ministério Público na condição de fiscal da ordem jurídica ."
]

results = predict_entities_batch(texts, model, t.tokenizer)
for res in results:
    print(res)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.
- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.
- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.
- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      

segue o texto
O ministério público acatou a decisão do STF e pediu a suspensão do processo contra o ex-presidente Lula, com base na Lei 17.681/2017.
Resposta:
O:O
 ministério:O
 público:O
 acatou:O
 a:O
 decisão:O
 do:O
 STF:O
 e:O
 pediu:O
 a:O
 suspensão:O
 do:O
 process

In [14]:
data

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1390
    })
})

In [8]:
len(LenerDataset(tokenizer=t.tokenizer).load_dataset()['train']['input_ids'][0])

INFO:src.datasets.lener:Loading dataset: peluz/lener_br
INFO:src.datasets.lener:Dataset loaded with splits: ['train', 'validation', 'test']
INFO:src.datasets.lener:Processing splits: ['train', 'validation', 'test']
INFO:src.datasets.lener:Original columns to remove after mapping: ['id', 'tokens', 'ner_tags']
INFO:src.datasets.lener:NER tag mapping created: {0: 'O', 1: 'B-ORGANIZACAO', 2: 'I-ORGANIZACAO', 3: 'B-PESSOA', 4: 'I-PESSOA', 5: 'B-TEMPO', 6: 'I-TEMPO', 7: 'B-LOCAL', 8: 'I-LOCAL', 9: 'B-LEGISLACAO', 10: 'I-LEGISLACAO', 11: 'B-JURISPRUDENCIA', 12: 'I-JURISPRUDENCIA'}
INFO:src.datasets.lener:Starting dataset mapping...
INFO:src.datasets.lener:Dataset mapping finished.
INFO:src.datasets.lener:Columns after mapping: ['id', 'tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels']


512

In [14]:
data['validation']['input_ids'][30]

[69286,
 3958,
 4443,
 32297,
 9087,
 16037,
 86020,
 3955,
 141402,
 4154,
 3524,
 23244,
 1197,
 13596,
 976,
 1467,
 436,
 13,
 1789,
 2121,
 1197,
 13596,
 1709,
 24709,
 36144,
 3524,
 23244,
 29610,
 1447,
 12,
 2726,
 58487,
 2843,
 135734,
 25,
 8550,
 485,
 7806,
 264,
 1197,
 13596,
 1709,
 4009,
 309,
 2872,
 17090,
 15249,
 11,
 7953,
 50304,
 11,
 7759,
 1963,
 15249,
 2569,
 2838,
 2782,
 11,
 506,
 6140,
 82,
 11,
 4992,
 624,
 12,
 393,
 9996,
 41339,
 25,
 6982,
 64,
 1197,
 13596,
 1709,
 29610,
 9662,
 288,
 409,
 45962,
 64846,
 15185,
 624,
 12,
 75670,
 2045,
 25,
 2876,
 924,
 1197,
 13596,
 1709,
 3158,
 309,
 64066,
 18965,
 2782,
 11,
 7953,
 16879,
 11,
 4812,
 37085,
 11,
 76082,
 16385,
 11,
 4992,
 624,
 12,
 42501,
 25,
 2263,
 3001,
 1197,
 13596,
 1709,
 4009,
 309,
 92824,
 3893,
 82481,
 16627,
 11,
 7953,
 272,
 13596,
 11,
 70877,
 11,
 93524,
 11,
 835,
 485,
 47919,
 11,
 4992,
 624,
 12,
 35426,
 1637,
 17845,
 74634,
 25,
 22507,
 29488,
 1197,


In [11]:
t.tokenizer.encode('''Texto: Nos termos do art . 114 , I , da Constituição da República , a Justiça do Trabalho afigura-se competente para examinar os litígios decorrentes da relação de trabalho , tenham eles fundo contratual ou não .
Resposta:
 LEGISLACAO: art . 114 , I , da Constituição da República<|im_end|>''')

[94923,
 25,
 49997,
 4647,
 436,
 653,
 1947,
 659,
 220,
 16,
 16,
 19,
 1154,
 358,
 1154,
 2994,
 75604,
 77023,
 2994,
 136962,
 1154,
 264,
 140233,
 653,
 1163,
 62794,
 6161,
 264,
 904,
 5690,
 7806,
 4533,
 6817,
 3348,
 7006,
 13762,
 2643,
 13020,
 70337,
 3530,
 10576,
 7976,
 288,
 2994,
 96103,
 409,
 54639,
 1154,
 5779,
 5604,
 66441,
 3802,
 78,
 87003,
 928,
 5908,
 12393,
 16448,
 1061,
 38531,
 510,
 35426,
 1637,
 43,
 1706,
 18746,
 25,
 1947,
 659,
 220,
 16,
 16,
 19,
 1154,
 358,
 1154,
 2994,
 75604,
 77023,
 2994,
 136962,
 151645]

In [30]:
print(t.tokenizer.decode(
    data['validation']['input_ids'][158],
    add_special_tokens=True
))

Você é um especialista jurídico responsável por identificar entidades em textos.        
As entidades que você deve identificar são:

- ORGANIZAÇÃO: Refere-se a entidades que representam organizações, como empresas, instituições governamentais, ONGs, etc.
- PESSOA: Designa entidades que são nomes de pessoas físicas.
- TEMPO: Marca entidades que expressam informações temporais, como datas, horários, períodos, etc.
- LOCAL: Indica entidades que representam lugares geográficos, como cidades, países, estados, endereços, etc.
- LEGISLAÇÃO: Identifica entidades que correspondem a Atos de Lei, como leis, decretos, portarias, etc.
- JURISPRUDÊNCIA: Assinala entidades que se referem a decisões relativas a casos legais.      

segue o texto
Texto: 2 Documento assinado digitalmente conforme MP n° 2.200-2/2001 de 24/08/2001 , que institui a Infraestrutura de Chaves Públicas Brasileira - ICP-Brasil .
Resposta:
2:O
Documento:O
assinado:O
digitalmente:O
conforme:O
MP:B-LEGISLACAO
n°:I-LEGISLACAO
2.20